# Qualitative analysis

This is to analyse which image regions the geolocation models consider the strongest, on application of different filters.

## Installations

In [ ]:
%pip install --quiet geoclip
%pip install pilgram

In [ ]:
from pathlib import Path
import os
from tqdm import tqdm
from glob import glob
import pandas as pd
import numpy as np
from geoclip import GeoCLIP
import torch

In [ ]:
from PIL import Image
import torchvision.transforms.functional as TF 
import torch.nn.functional as F
from torchvision.transforms import GaussianBlur

## Setup

In [ ]:
os.makedirs("./results", exist_ok=True)

## Load GeoCLIP model

In [ ]:
model = GeoCLIP()
model.to("cuda:0")

In [ ]:
import torch.nn.functional as F
from transformers.modeling_outputs import ModelOutput

# Kaggle's transformers returns a ModelOutput from CLIP's vision tower that geoclip
# forgets to unwrap before its Linear layers; patch F.linear to unwrap it defensively.
_orig_linear = F.linear

def _patched_linear(input, weight, bias=None):
    if isinstance(input, ModelOutput):
        pooled = input.get("pooler_output", None)
        input = pooled if pooled is not None else input.get("last_hidden_state")
    return _orig_linear(input, weight, bias)

F.linear = _patched_linear

## Function to perform square masking

In [ ]:
def load_image(image_path):
    # Resize image to 512x512
    image = Image.open(image_path).convert('RGB')
    image = image.resize((512, 512))
    return image

from PIL import Image, ImageFilter


def occlude_one_patch(
    image,
    row,
    col,
    grid_size=8,
    sigma=15.0,
):
    """
    Blur one region of a PIL image.

    Args:
        image: PIL.Image.Image
        row: row index of the region
        col: column index of the region
        grid_size: number of regions along each dimension
        sigma: Gaussian blur radius

    Returns:
        PIL.Image.Image with the selected region blurred.
    """

    image = image.copy()

    width, height = image.size

    cell_w = width // grid_size
    cell_h = height // grid_size

    x1 = col * cell_w
    y1 = row * cell_h

    # Make sure the final row/column includes any remaining pixels
    x2 = (col + 1) * cell_w if col < grid_size - 1 else width
    y2 = (row + 1) * cell_h if row < grid_size - 1 else height

    # Crop the selected region
    patch = image.crop((x1, y1, x2, y2))

    # Gaussian blur
    blurred_patch = patch.filter(
        ImageFilter.GaussianBlur(radius=sigma)
    )

    # Put blurred patch back into image
    image.paste(
        blurred_patch,
        (x1, y1)
    )


    return image, (x1, y1, x2, y2)

## Function to get GeoCLIP outputs

In [ ]:
def get_geoclip_outputs(image):
    image = model.image_encoder.preprocess_image(image)
    image = image.to(model.device)

    gps_gallery = model.gps_gallery.to(model.device)

    logits_per_image = model.forward(image, gps_gallery)
    probs_per_image = logits_per_image.softmax(dim=-1).cpu()

    # Get top 1 predictions
    top_pred = torch.topk(probs_per_image, 1, dim=1)
    top_pred_gps = model.gps_gallery[top_pred.indices[0]]
    top_pred_prob = top_pred.values[0]

    # Convert to (lat, lon) format
    lat, lon = top_pred_gps[0]
    lat = float(lat.item())
    lon = float(lon.item())
    return (lat, lon)


## Function to get Haversine distance

In [ ]:
from haversine import haversine, Unit

def compute_haversine_distance(gt_coords, comp_coords):
    dist = haversine(gt_coords, comp_coords, unit=Unit.KILOMETERS)
    return dist

## Apply patch occlusion and get model outputs for images

**Note**: Update the path here

In [ ]:
all_images = glob("path_to_datset/*.jpg") # Update the path here

In [ ]:
from glob import glob
from tqdm import tqdm
from pathlib import Path

GRID_SIZE = 64

all_image_paths = []
all_raw_predictions = []
all_comp_predictions = []
all_comp_coords = []

for image_path in tqdm(all_images, desc="Images progress"):
    # Get first prediction
    base_image = load_image(image_path)
    raw_prediction = get_geoclip_outputs(base_image)

    # Save raw prediction
    all_image_paths.append(image_path)
    all_raw_predictions.append(raw_prediction)
    all_comp_predictions.append(raw_prediction)
    all_comp_coords.append(None)

    # Get all predictions for patches
    all_coordinates = []
    for X in range(GRID_SIZE):
        for Y in range(GRID_SIZE):
            comp_image, blurred_coords = occlude_one_patch(base_image, X, Y, GRID_SIZE)
            comp_predictions = get_geoclip_outputs(comp_image)

            # Save prediction
            all_image_paths.append(image_path)
            all_raw_predictions.append(raw_prediction)
            all_comp_predictions.append(comp_predictions)
            all_comp_coords.append(blurred_coords)

            all_coordinates.append(comp_predictions)

    # Print the number of coordinates
    print(image_path)
    print(len(set(all_coordinates)))

    # Clear cache
    torch.cuda.empty_cache()

# Create dataframe
df = pd.DataFrame({
    "image_path" : all_image_paths,
    "raw_prediction" : all_raw_predictions,
    "comp_prediction" : all_comp_predictions,
    "blur_coordinates" : all_comp_coords
})

df.to_csv(f"./results/geoclip_{GRID_SIZE}_multibluroutputs.csv")

## Apply an image filter and run the analysis

**Note**: The Willow filter is used in this cell, it can be changed to any PILGram-compatible filter

In [ ]:
from glob import glob
from tqdm import tqdm
from pathlib import Path
import pilgram

all_images = MULTI_OUTPUT_FILES
GRID_SIZE = 64

all_image_paths = []
all_raw_predictions = []
all_comp_predictions = []
all_comp_coords = []

for image_path in tqdm(all_images, desc="Images progress"):
    # Get first prediction
    base_image = load_image(image_path)

    # Apply filter
    base_image = pilgram.willow(base_image)
    
    raw_prediction = get_geoclip_outputs(base_image)

    # Save raw prediction
    all_image_paths.append(image_path)
    all_raw_predictions.append(raw_prediction)
    all_comp_predictions.append(raw_prediction)
    all_comp_coords.append(None)

    # Get all predictions for patches
    all_coordinates = []
    for X in range(GRID_SIZE):
        for Y in range(GRID_SIZE):
            comp_image, blurred_coords = occlude_one_patch(base_image, X, Y, GRID_SIZE)
            comp_predictions = get_geoclip_outputs(comp_image)

            # Save prediction
            all_image_paths.append(image_path)
            all_raw_predictions.append(raw_prediction)
            all_comp_predictions.append(comp_predictions)
            all_comp_coords.append(blurred_coords)

            all_coordinates.append(comp_predictions)

    # Print the number of coordinates
    print(image_path)
    print(len(set(all_coordinates)))

    # Clear cache
    torch.cuda.empty_cache()

# Create dataframe
df = pd.DataFrame({
    "image_path" : all_image_paths,
    "raw_prediction" : all_raw_predictions,
    "comp_prediction" : all_comp_predictions,
    "blur_coordinates" : all_comp_coords
})

df.to_csv(f"willow_geoclip_{GRID_SIZE}_multibluroutputs.csv")

# Compute and save importance heatmaps

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image


def show_importance_heatmap(
    filter_name,
    image_path,
    ax,
    image_type="multi",
    distance_column="haversine_distance",
    image_size=(512, 512),
    alpha=0.60,
    grid_size=64
):
    # Load filter
    df = pd.read_csv(f"./results/{filter_name}_geoclip_{grid_size}_{image_type}bluroutputs.csv")
    
    # Get only the patches belonging to this image
    image_df = df[
        df["image_path"] == image_path
    ].copy()

    # Remove the raw/unmodified image row
    image_df = image_df[
        image_df["blur_coordinates"].notna()
    ]

    # Create a distance column
    image_df['raw_prediction'] = image_df.apply(lambda x: eval(x['raw_prediction']), axis=1)
    image_df['comp_prediction'] = image_df.apply(lambda x: eval(x['comp_prediction']), axis=1)
    image_df[distance_column] = image_df.apply(lambda x: compute_haversine_distance(x["raw_prediction"], x["comp_prediction"]), axis=1)


    # Create 512x512 importance map
    importance = np.zeros(
        (image_size[1], image_size[0]),
        dtype=np.float32
    )

    # Get distances
    distances = image_df[
        distance_column
    ].astype(float).values

    # Normalize distances to 0-1
    min_dist = distances.min()
    max_dist = distances.max()

    if max_dist > min_dist:
        scores = (
            distances - min_dist
        ) / (
            max_dist - min_dist
        )
    else:
        scores = np.zeros_like(distances)

    # Put each score into its corresponding patch
    for (_, row), score in zip(
        image_df.iterrows(),
        scores
    ):
        x1, y1, x2, y2 = eval(row["blur_coordinates"])

        importance[
            y1:y2,
            x1:x2
        ] = score

    # Load image
    image = Image.open(image_path).convert("RGB")

    # Plot
    plt.figure(figsize=(8, 8))

    ax.imshow(image)

    heatmap = ax.imshow(
        importance,
        cmap="jet",
        alpha=alpha,
        vmin=0,
        vmax=1
    )

    ax.axis("off")

    return heatmap


## Saving the heatmaps into a unified figure

**Note**: This is enabled when individual heatmaps are computed for different filters, and one figure is produced for an input image

In [ ]:
fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 5)
)

image_path = all_images[0]

heatmap1 = show_importance_heatmap("colour", image_path, axes[0], "multi")
heatmap2 = show_importance_heatmap("clarendon", image_path, axes[1], "multi")
heatmap3 = show_importance_heatmap("willow", image_path, axes[2], "multi")


axes[0].set_title("Original image")
axes[1].set_title("Clarendon filter (colour)")
axes[2].set_title("Willow filter (black-and-white")

# One shared colorbar
fig.colorbar(
    heatmap1,
    ax=axes,
    fraction=0.025,
    pad=0.02,
    label="Prediction displacement"
)

plt.tight_layout()
# plt.show()
fig.savefig("./results/filter_comparison_figure.jpg")